# Lab 5 — Project Setup & Memory (Features #1–#5)
**AmbariCloud FDE Workshop · Day 1** · Work inside `~/workshop-todo` from Lab 3.

## Feature #1 — CLAUDE.md: your project brain
- Auto-loads EVERY session. Put: tech stack, build/test commands, folder map, conventions, rules.
- It **advises** — instructions, not law. Hard guarantees belong in **hooks** (notebook 03).
- Keep it short: under ~100 lines; identity + commands only.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> /memory        # open ./CLAUDE.md, add:
  ## Commands
  - test: pytest -q
  - run:  python todo.py list
  ## Rules
  - Always run pytest before saying done

> /clear
> add a 'clear' command that empties the list
```
**Should see:** Claude runs pytest at the end WITHOUT being asked — CLAUDE.md said so.
**Watch for:** a rule Claude ignores is a rule too long or too vague — one imperative line, or move to a hook.


## Feature #2 — CLAUDE.md hierarchy: most specific wins
1. **Enterprise** — managed policy (IT-controlled)
2. **User** — `~/.claude/CLAUDE.md` (your habits, every project)
3. **Project** — `./CLAUDE.md` (shared via git); `CLAUDE.local.md` = private, auto-gitignored
4. **Subdirectory** — `src/api/CLAUDE.md` (loaded when Claude touches files there)
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```bash
echo '- Prefer short answers' >> ~/.claude/CLAUDE.md
mkdir -p tests && cat > tests/CLAUDE.md <<'EOF'
Tests use pytest; name files test_*.py;
never mock the JSON store, use tmp_path.
EOF
```
Then `> /memory` shows all layers. Ask `add a test for done` → Claude follows `tests/CLAUDE.md` (tmp_path) even though `./CLAUDE.md` never mentioned it.
**Watch for:** secrets and personal paths go in `CLAUDE.local.md`, never the shared file.


### Verify hierarchy layers exist (runnable)

In [ ]:
import os, pathlib
for p in ['~/.claude/CLAUDE.md', '~/workshop-todo/CLAUDE.md', '~/workshop-todo/tests/CLAUDE.md']:
    f = pathlib.Path(os.path.expanduser(p))
    print(('FOUND ' if f.exists() else 'missing') , p)

## Feature #3 — /init: onboarding in 90 seconds
Scans tree, manifests, README, CI files → writes a starter CLAUDE.md. Output is a **draft**: trim generic text.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```bash
git clone https://github.com/pallets/flask /tmp/flask
cd /tmp/flask && claude
> /init        # ~60-90s: detects Python, pyproject, tox, src layout
> cat CLAUDE.md
```
Same command, wildly different output depending on repo size — value scales with codebase.
**Watch for:** `/init` describes what IS there; team conventions in people's heads must be added by hand (or `#`, next).


## Feature #4 — `#`: quick add to memory
Start a line with `#` → text is saved to a CLAUDE.md file. Picker asks WHICH: Project / User / Local.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> # Always run pytest -q before claiming a task is done      -> choose Project
> # I prefer type hints on every function                     -> choose User
> /memory      # both visible
> /clear
> add a 'count' command       # type hints appear — memory survived
```
**Watch for:** `#` is NOT a comment — it's a memory shortcut. Not for secrets (plain text in repo).


## Feature #5 — `@`: file mention
`@path` (Tab autocompletes) pulls a file or directory INTO context. Directory = listing; file = full content. Also MCP resources: `@server:resource`.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> explain @todo.py line by line
> compare @todo.py with @tests/test_todo.py — which command has no test?
> @tests/         # tree only
> /context        # see the @ files counted
```
**Should see:** answers with exact line numbers (`todo.py:12`) — it READ the file, didn't guess.
**Watch for:** every `@` costs context for the whole session until `/clear` or `/compact`.


### Headless equivalent (runnable) — @-mention via prompt

In [ ]:
%%bash
cd ~/workshop-todo 2>/dev/null || { echo "create ~/workshop-todo first (Lab 3)"; exit 0; }
claude -p "Read todo.py and list its CLI commands, one per line" --allowedTools Read 2>&1 | tail -10